# Day 5 — Functions, args/kwargs, scope, docstrings, type hints
Objectives:
- Define functions with defaults, *args, **kwargs.
- Document with docstrings.
- Add typing for clarity and tooling support.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-05`. Read
`python/ds-60day/companion-guides/day05_functions_type_hints.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A function gives a name to a reusable behavior. Its contract says which
inputs are accepted, what it returns, which failures it raises, and
whether it changes anything outside itself. Parameters are names in the
definition; arguments are actual values supplied by a caller.

A call creates a local scope. Names assigned there normally disappear
when the call returns. A `return` sends one value to the caller and
stops that call. Type hints document intended types and support static
tools, but Python does not enforce them automatically at runtime.
Defaults are evaluated once when `def` runs, so mutable defaults should
normally be replaced by `None` plus a fresh object inside the function.

### Vocabulary

- **function:** a named reusable block that can accept inputs and return a value.
- **parameter:** a name declared in a function signature.
- **argument:** a value supplied for a parameter during a call.
- **return value:** the object sent back to the caller.
- **scope:** the region in which a name can be resolved.
- **type hint:** machine-readable documentation of an intended type.

## Syntax anatomy

`def clamp(value: float, *, low: float, high: float) -> float:` begins
with `def`, gives the function a name, annotates three parameters, uses
`*` to make `low` and `high` keyword-only, and annotates the return.
The colon starts the indented body. A docstring is the first string in
that body and should state behavior that the signature cannot fully
express, especially errors and edge cases.

### Worked example 1 — Make validation part of the contract

Reject an invalid domain before doing the calculation. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
def mean(values: list[float]) -> float:
    """Return the arithmetic mean of a non-empty list."""
    if not values:
        raise ValueError("values must not be empty")
    return sum(values) / len(values)

mean([2.0, 4.0, 9.0])

**Expected observation:** `5.0`. Empty input follows a deliberate exception path instead of dividing by zero accidentally.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Use keyword-only parameters to make calls readable

A signature can prevent ambiguous positional calls. Predict first; then run the next cell.

In [ ]:
def discounted(price: float, *, rate: float = 0.0) -> float:
    if not 0 <= rate <= 1:
        raise ValueError("rate must be between 0 and 1")
    return price * (1 - rate)

discounted(80.0, rate=0.25)

**Expected observation:** `60.0`. The call labels `rate`, making `0.25` hard to confuse with another quantity.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Read the signature and call side by side when arguments bind unexpectedly.
2. Check whether every path reaches an explicit `return`; falling off the end returns `None`.
3. Replace a mutable default such as `items=[]` with `items: list[...] | None = None`.
4. Use a type checker for hints, but still validate external runtime data at the boundary.

**Alternative to compare:** A tuple can return several related values; a dataclass becomes clearer when those values need durable names and behavior.

**Boundary to test:** Empty input, mismatched lengths, zero total weight, invalid numeric ranges, and Unicode/whitespace normalization need written policy.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
from typing import List, Dict, Any

def summarize(title: str, *items: str, uppercase: bool = False, **meta: Any) -> Dict[str, Any]:
    """Summarize a list of items.
    Args:
        title: heading for the summary
        *items: arbitrary number of items
        uppercase: if True, convert items to upper
        **meta: arbitrary metadata
    Returns: dict with summary.
    """
    data: List[str] = [i.upper() if uppercase else i for i in items]
    return {"title": title, "count": len(data), "items": data, "meta": meta}

summarize('Todo', 'clean', 'analyze', uppercase=True, owner='you')


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Implement `describe(values: list[float]) -> tuple[float, float]` returning the arithmetic mean and **population** standard deviation. **Input:** `[2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]`.
   **Expected result:** `(5.0, 2.0)`. **Constraints:** compute the mean once, use squared distances, and do not import a statistics helper that performs the whole task.
   **Verify:** Use `math.isclose` to confirm mean `5.0` and population standard deviation `2.0`, then show empty input follows the documented failure path.

2. Add a docstring to `describe` that states accepted input, the two tuple fields in order, and the empty-input policy. **Constraint:** choose and document one explicit behavior—this lesson's reference raises `ValueError`.
   **Verify:** capture `help(describe)` output and assert it names the accepted input, mean, population standard deviation, tuple order, and empty-input `ValueError` without reading the body.

3. Validate `describe` near its boundary: reject an empty list and any non-finite value such as `float('nan')` with a useful `ValueError`.
   **Verify:** show the normal result and use two separate `try`/`except ValueError` checks for the invalid cases; do not catch errors inside the function that it cannot repair.

### Additional mastery practice

Design functions from their contracts: accepted inputs, return value, failure behavior, side effects, and boundary cases.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** Predict the result of calling a function with `items=[]` as a default three times when it appends on each call. Explain shared defaults.
   **Progressive hint:** Default objects are created once when `def` executes.
   **Verify:** Call the faulty function three times and record cumulative state, then assert the `None`-sentinel repair returns independent one-item lists on every call.
5. **Tracing:** Trace a local variable that shadows a global of the same name. Which binding changes, and when would `global` be required?
   **Progressive hint:** Assignment makes a name local unless explicitly declared otherwise.
   **Verify:** Record local/global values before, during, and after the call; confirm ordinary local assignment leaves the global unchanged.
6. **Implementation:** Implement typed `weighted_mean(values, weights)` with length, empty, and zero-total-weight validation.
   **Progressive hint:** State every invalid condition before calculating.
   **Verify:** Assert a known weighted mean and separately assert empty, length-mismatch, and zero-total-weight inputs raise the documented errors.
7. **Debugging:** Repair a function whose `*items` argument is accidentally passed as one list instead of unpacked individual items.
   **Progressive hint:** Compare `f(values)` with `f(*values)`.
   **Verify:** Capture arguments received by `f(values)` and `f(*values)`; assert the repaired call presents individual items rather than one nested list.
8. **Edge case and explanation:** Write a docstring for a name-normalization function and test empty, whitespace-only, and Unicode input.
   **Progressive hint:** Document whether empty normalized output is valid or an error.
   **Verify:** Test ordinary, empty, whitespace-only, and Unicode names against the docstring's stated contract; every behavior must match the documentation.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Implement `describe(values: list[float]) -> tuple[float, float]` returning the arithmetic mean and **population** standard deviation. **Input:** `[2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]`. **Expected result:** `(5.0, 2.0)`. **Constraints:** compute the mean once, use squared distances, and do not import a statistics helper that performs the whole task. **Verify:** Use `math.isclose` to confirm mean `5.0` and population standard deviation `2.0`, then show empty input follows the documented failure path.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Implement `describe(values: list[float]) -> tuple[float, float]` returning the arithmetic mean and **population** standard deviation. `[2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]`....
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add a docstring to `describe` that states accepted input, the two tuple fields in order, and the empty-input policy. **Constraint:** choose and document one explicit behavior—this lesson's reference raises `ValueError`. **Verify:** capture `help(describe)` output and assert it names the accepted input, mean, population standard deviation, tuple order, and empty-input `ValueError` without reading the body.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add a docstring to `describe` that states accepted input, the two tuple fields in order, and the empty-input policy. choose and document one explicit behavior—this lesson's refe...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** Validate `describe` near its boundary: reject an empty list and any non-finite value such as `float('nan')` with a useful `ValueError`. **Verify:** show the normal result and use two separate `try`/`except ValueError` checks for the invalid cases; do not catch errors inside the function that it cannot repair.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Validate `describe` near its boundary: reject an empty list and any non-finite value such as `float('nan')` with a useful `ValueError`. show the normal result and use two separa...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the result of calling a function with `items=[]` as a default three times when it appends on each call. Explain shared defaults. **Progressive hint:** Default objects are created once when `def` executes. **Verify:** Call the faulty function three times and record cumulative state, then assert the `None`-sentinel repair returns independent one-item lists on every call.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Predict the result of calling a function with `items=[]` as a default three times when it appends on each call. Explain shared defaults. Default objects are created once when `d...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace a local variable that shadows a global of the same name. Which binding changes, and when would `global` be required? **Progressive hint:** Assignment makes a name local unless explicitly declared otherwise. **Verify:** Record local/global values before, during, and after the call; confirm ordinary local assignment leaves the global unchanged.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Trace a local variable that shadows a global of the same name. Which binding changes, and when would `global` be required? Assignment makes a name local unless explicitly declar...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement typed `weighted_mean(values, weights)` with length, empty, and zero-total-weight validation. **Progressive hint:** State every invalid condition before calculating. **Verify:** Assert a known weighted mean and separately assert empty, length-mismatch, and zero-total-weight inputs raise the documented errors.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Implement typed `weighted_mean(values, weights)` with length, empty, and zero-total-weight validation. State every invalid condition before calculating. Assert a known weighted...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a function whose `*items` argument is accidentally passed as one list instead of unpacked individual items. **Progressive hint:** Compare `f(values)` with `f(*values)`. **Verify:** Capture arguments received by `f(values)` and `f(*values)`; assert the repaired call presents individual items rather than one nested list.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Repair a function whose `*items` argument is accidentally passed as one list instead of unpacked individual items. Compare `f(values)` with `f(*values)`. Capture arguments recei...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Write a docstring for a name-normalization function and test empty, whitespace-only, and Unicode input. **Progressive hint:** Document whether empty normalized output is valid or an error. **Verify:** Test ordinary, empty, whitespace-only, and Unicode names against the docstring's stated contract; every behavior must match the documentation.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Write a docstring for a name-normalization function and test empty, whitespace-only, and Unicode input. Document whether empty normalized output is valid or an error. Test ordin...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
